In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Ready for feature selection!")

✅ Ready for feature selection!


In [ ]:
from google.colab import files
import pandas as pd # Import pandas
uploaded = files.upload()
import io
df = pd.read_csv(io.BytesIO(uploaded[list(uploaded.keys())[0]]))
print("Your file loaded!")


Saving diabetes_cleaned (1).csv to diabetes_cleaned (1) (1).csv
✅ Your file loaded!


In [ ]:
def advanced_feature_engineering(df):
    """
    Create new features and select most important ones
    """
    df_engineered = df.copy()

    # 1. Create new interaction features
    print("🔧 Creating new features...")

    # BMI categories
    df_engineered['bmi_category'] = pd.cut(df_engineered['bmi'],
                                         bins=[0, 18.5, 25, 30, 100],
                                         labels=['Underweight', 'Normal', 'Overweight', 'Obese'])

    # Age groups
    df_engineered['age_group'] = pd.cut(df_engineered['age'],
                                      bins=[0, 30, 45, 60, 100],
                                      labels=['Young', 'Middle', 'Senior', 'Elderly'])

    # Risk score (composite feature)
    df_engineered['risk_score'] = (
        df_engineered['HbA1c_level'] * 0.4 +
        df_engineered['blood_glucose_level'] * 0.3 +
        df_engineered['bmi'] * 0.2 +
        df_engineered['age'] * 0.1
    )

    # 2. Polynomial features (degree 2)
    df_engineered['bmi_squared'] = df_engineered['bmi'] ** 2
    df_engineered['age_squared'] = df_engineered['age'] ** 2
    df_engineered['hba1c_squared'] = df_engineered['HbA1c_level'] ** 2

    # 3. Encode new categorical features
    df_engineered = pd.get_dummies(df_engineered, columns=['bmi_category', 'age_group'], drop_first=True)

    # One-hot encode original categorical features
    df_engineered = pd.get_dummies(df_engineered, columns=['gender', 'smoking_history'], drop_first=True)


    print(f"Feature engineering completed! New shape: {df_engineered.shape}")
    return df_engineered

def feature_selection(X, y, k=15):
    """
    Select most important features using multiple methods
    """
    from sklearn.feature_selection import SelectKBest, f_classif, RFE
    from sklearn.ensemble import RandomForestClassifier

    print("Performing feature selection...")

    # Method 1: SelectKBest (ANOVA F-value)
    selector_kbest = SelectKBest(score_func=f_classif, k=k)
    X_kbest = selector_kbest.fit_transform(X, y)
    selected_features_kbest = X.columns[selector_kbest.get_support()].tolist()

    # Method 2: Recursive Feature Elimination (RFE)
    model_rfe = RandomForestClassifier(random_state=42)
    selector_rfe = RFE(estimator=model_rfe, n_features_to_select=k)
    X_rfe = selector_rfe.fit_transform(X, y)
    selected_features_rfe = X.columns[selector_rfe.get_support()].tolist()

    # Method 3: Random Forest Feature Importance
    model_rf = RandomForestClassifier(random_state=42)
    model_rf.fit(X, y)
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': model_rf.feature_importances_
    }).sort_values('importance', ascending=False)

    selected_features_rf = feature_importance.head(k)['feature'].tolist()

    # Combine results
    all_selected = list(set(selected_features_kbest + selected_features_rfe + selected_features_rf))

    print(f"Selected {len(all_selected)} features:")
    for feature in all_selected:
        print(f"  - {feature}")

    return all_selected, feature_importance

# Apply feature engineering
df_engineered = advanced_feature_engineering(df)

# Prepare for feature selection
X_eng = df_engineered.drop('diabetes', axis=1)
y_eng = df_engineered['diabetes']

# Perform feature selection
selected_features, importance_df = feature_selection(X_eng, y_eng, k=10)

# Use only selected features
X_selected = X_eng[selected_features]

print("\nFeature Importance Ranking:")
print(importance_df.head(10))

🔧 Creating new features...
✅ Feature engineering completed! New shape: (55046, 30)
🎯 Performing feature selection...
📊 Selected 12 features:
  - hypertension
  - age_group_Elderly
  - risk_score
  - smoking_history_never
  - bmi
  - hba1c_squared
  - age
  - age_squared
  - bmi_squared
  - blood_glucose_level
  - HbA1c_level
  - gender_Male

🔍 Feature Importance Ranking:
                feature  importance
15        hba1c_squared    0.214082
12           risk_score    0.201555
4           HbA1c_level    0.190789
5   blood_glucose_level    0.147236
13          bmi_squared    0.049838
3                   bmi    0.049219
0                   age    0.039588
14          age_squared    0.038562
1          hypertension    0.009653
22          gender_Male    0.008728
